# Idea 2: Snooze Club

Recompute the evidence behind the pop-up registration proposal and targeted VIP
voucher test from the verified shared dataset. This notebook covers newcomer
code routes, website value/return cohorts, point-funded vouchers, member-linked
purchase gaps and illustrative break-even economics.

Before starting Jupyter, set `WEAVVE_DATA` to the original files and `WEAVVE_OUT`
to a private output folder, then run notebook 04 or `python scripts/build_clean_data.py`.
Optionally select an existing verified dataset with `WEAVVE_CLEAN`. The report
snapshot ends on **28 August 2026**; other cutoffs are rejected.

Run all cells from inside this repository. Restart the kernel after changing
data paths or code. Keep executed notebooks and results outside Git.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo = next((p for p in (cwd, *cwd.parents)
             if (p / 'src/weavve/final/snooze_club.py').is_file()), None)
if repo is None:
    raise RuntimeError('Start Jupyter inside this repository.')
sys.path.insert(0, str(repo / 'src'))
from weavve.clean import load_dataset
from weavve.final.snooze_club import build_evidence, voucher_sensitivity

dataset = load_dataset()
report = build_evidence(dataset)
print('Dataset:', report['dataset_id'])
print('Transaction cutoff:', report['transaction_cutoff'])
display(pd.DataFrame(report['source_tables']).T)

## 1. Pop-up and club code routes

A newcomer has their first observed positive website/Shopee purchase on the
website. This is not a verified lifetime first purchase. Compare 28 Aug 2023 to
26 Aug 2024 with 27 Aug 2024 to 26 Aug 2025.

Groups are mutually exclusive: club code first, then pop-up-related code, other
recorded discount/code, then no recorded discount/code. Repeated monetary rows
can recover code annotations but never add sales. The second calculation uses
retained lines only, matching the earlier sensitivity definition.

Code use is not completed registration. Associated first-order sales are not
incremental pop-up revenue; these data have no visitor/exposure denominator.

In [ ]:
popup_rows = []
for method, windows in report['popup'].items():
    for window, values in windows.items():
        for route, measures in values['groups'].items():
            popup_rows.append(dict(method=method, window=window,
                                   start=values['start'], end=values['end'],
                                   newcomers=values['newcomers'], route=route, **measures))
display(pd.DataFrame(popup_rows))

## 2. High-value website buyers and next-year return

Rank all eligible website buyers by annual website net sales. Select
`ceil(10% of buyers)` with deterministic person-ID tie resolution. These are
website spending groups, not verified club members.

Keep value concentration in Sep 2024 to Aug 2025 separate from the Sep 2023 to
Aug 2024 cohort's following-year return. The newer cohort's follow-up ends on
28 Aug 2026, three days before the end of its next calendar year. Website return
does not establish permanent churn or purchases through other channels.

In [ ]:
value_rows = []
for cohort, values in report['website_top_deciles'].items():
    for group, measures in values['groups'].items():
        value_rows.append(dict(cohort=cohort, group=group,
                               baseline_start=values['baseline_start'],
                               baseline_end=values['baseline_end'],
                               followup_end=values['followup_end'],
                               full_followup=values['full_next_calendar_year_observed'], **measures))
display(pd.DataFrame(value_rows))

## 3. Point-funded cash voucher use

Select negative-point reward records whose descriptions identify a cash
voucher. Rows are voucher records, not unique members. Used status and reward
selection dates do not establish the date or incremental effect of redemption.
Higher use among larger denominations does not prove that a larger voucher
caused more spending.

In [ ]:
reward = report['rewards_and_cadence']
display(pd.DataFrame(reward['cash_voucher_records']))
selected = reward['cash_point_funded_records']
used = reward['used_cash_point_funded_records']
print('Point-funded reward records:', reward['point_funded_records'])
print('Cash voucher records:', selected, '| Recorded used:', used)
print('Recorded use rate:', f'{used / selected:.1%}' if selected else 'Not available')

## 4. Historical member-linked purchase timing

Membership evidence is historical club-code use or a point-funded reward linked
by website order or exact retained sale code. It is not a complete current
membership or contact-permission roster. Rank linked buyers active from
29 Aug 2025 to 28 Aug 2026 using spending in that window.

Purchase gaps use each group's full website history. Orders within seven days
of an occasion's start are consolidated. Show the median of personal median
gaps alongside the pooled median: they answer different questions. These are
descriptive timing hypotheses, not an optimal voucher frequency or causal
effect. Current spend rank must not be used as a pre-treatment experiment rank.

In [ ]:
print('Active ranking window:', ' to '.join(reward['active_window']))
display(pd.DataFrame(reward['all_website_history_of_latest_active_members']).T)

## 5. Illustrative additional-voucher break-even

The following inputs are **planning assumptions, not a forecast or an eligible
audience estimate**. Assume 100 redemptions, S$200 baskets before the extra
reward, a 10% extra voucher capped at S$20 and S$500 setup/campaign cost.
Compare contribution margins of 40%, 50% and 60% before the extra reward.

`net gain = additional orders × basket × margin − all redemptions × capped discount − setup`

All redeemed test vouchers cost money, including on purchases that would have
happened anyway. Zero additional orders is included. Other variable costs are
assumed included in margin. Equal baskets, displaced offers, future purchase
timing and recurring work require separate checks before a business decision.
These cases have no annual recurrence assumption.

In [ ]:
illustration = voucher_sensitivity(
    redemptions=100, basket=200, discount_rate=.10, cap=20, setup=500,
    margins=(.4, .5, .6), additional_orders=(0, 20, 30),
)
display(pd.json_normalize(illustration))

## 6. Interpretation and scope

- Test clearer registration with the same welcome offer and creative in both
  groups; measure membership, purchase conversion and contribution per assigned
  visitor. A shift in code choice alone does not establish growth.
- For a VIP test, verify rank, membership and permission before allocation.
  Compare the same reminder with and without one additional voucher; analyse
  contribution per assigned member, including non-redeemers.
- Avoid overlapping Idea 1 messages and counting shared costs or purchases twice.
- The archived second-to-third purchase comparison and pooled annual spending
  narrative are separate report sources; this notebook does not recompute them.
  No observed or assumed lift from those sources is inserted as a causal effect.

`analysis/idea2/idea2_size.py` remains the team's separate size-of-prize script
using cleaned Excel workbooks and its own assumptions. This notebook uses the
verified canonical tables and the fixed-redemption illustration above; the two
models are not interchangeable.

The displayed results stay in this session. Save an executed copy only in a
private folder outside the checkout. Commit this source notebook with empty
outputs, no attachments and no execution counts.